In [1]:
from pathlib import Path
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ruleset_comparison"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPERIMENT_ID = f"exp_ruleset_comparison_{RUN_TS}"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("EXPERIMENT_ID:", EXPERIMENT_ID)

PROJECT_ROOT: /home/harielpadillasanchez/Documentos/TT/TT2
DATA_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/data
OUTPUT_DIR: /home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison
EXPERIMENT_ID: exp_ruleset_comparison_20260322_133408


In [2]:
import json
import pandas as pd

from configs.models import MODELS
from configs.rules import RULESETS
from src.experiment.runner import ExperimentRunner
from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

/home/harielpadillasanchez/Documentos/TT/TT2/.venv-bloom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SAMPLE_PATH = DATA_DIR / "Muestra_csv.csv"

df_sample = pd.read_csv(SAMPLE_PATH)

print("Shape:", df_sample.shape)
print("Columnas:", list(df_sample.columns))
display(df_sample.head(3))

Shape: (20, 17)
Columnas: ['id', 'idFinal', 'grupo', 'motivo', 'Segmento', 'Propuesta', 'idcod', 'atr0', 'atr1', 'atr2', 'atr3', 'atr4', 'atr5', 'atr6', 'atr7', 'atr8', 'lex']


,id,idFinal,grupo,motivo,Segmento,Propuesta,idcod,atr0,atr1,atr2,atr3,atr4,atr5,atr6,atr7,atr8,lex
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,Vivian,10,1,4,5.0,NaN,NaN,NaN,NaN,NaN,1
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Vivian,14,5,1,6.0,NaN,NaN,NaN,NaN,NaN,1
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Vivian,5,6,19,1.0,NaN,NaN,NaN,NaN,NaN,1


In [4]:
META_COLS = ["idFinal", "grupo", "motivo", "lex"]

required_cols = ["id", "Segmento", "Propuesta"]
missing = [c for c in required_cols if c not in df_sample.columns]

if missing:
    raise ValueError(f"Faltan columnas requeridas en la muestra: {missing}")

df_refine = df_sample.copy()

df_refine = df_refine.rename(
    columns={
        "id": "sample_id",
        "Segmento": "source_text",
        "Propuesta": "reference_text",
    }
)

keep_cols = ["sample_id", "source_text", "reference_text"] + [c for c in META_COLS if c in df_refine.columns]
df_refine = df_refine[keep_cols].copy()

print("Shape refinamiento:", df_refine.shape)
display(df_refine.head(5))

Shape refinamiento: (20, 7)


,sample_id,source_text,reference_text,idFinal,grupo,motivo,lex
0,2088,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,2976,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,3430,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1
3,3679,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,1
4,3145,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,1


In [5]:
FEW_SHOT_EXAMPLES = [
    {
        "source": """A continuación una tabla donde se demuestra como crecen los ahorros al pasar los años: Edad Inversión Intereses Saldo Inversión Intereses Saldo Laura Ganados Alberto Ganados 22 $800 $80 $880 23 $800 $168 $1.232 Valor al Jubilarse $311.092 Valor al Jubilarse $263.232 Menos las contribuciones iniciales $6.400 Menos las contribuciones iniciales $28.800 Ganancia Neta $304.692 Ganancia Neta $234.""",
        "target": """A continuación, presentamos una tabla donde se demuestra como crecen los ahorros al pasar los años. Las categorías en que se divide la tabla son edad, inversión, intereses ganados y saldo. La tabla contrasta las ganancias en intereses por año de Laura y Alberto con una proyección de edad de jubilación de sesenta y cinco años y según la cantidad de años que mantuvieron el ahorro."""
    },
    {
        "source": """El varias veces mencionado Yager, nos dice acertadamente sobre estos aspectos lo siguiente: La mayoría de ustedes están hambrientos de tener libertad financiera, están hastiados de tener batallas financieras porque son como un carrusel.""",
        "target": """Yager habla acertadamente sobre estos aspectos y dice que la mayoría ambiciona tener libertad financiera y está cansada de tener batallas financieras inútiles."""
    },
    {
        "source": """De acuerdo con esta medición, la proporción de pobres en la población mundial quienes viven con menos de un dólar por día descendió levemente entre 1987 y 1993, pues pasó del 30% al 29%.""",
        "target": """Según esta medición, la proporción de personas pobres en la población mundial descendió un poco entre mil novecientos ochenta y siete y mil novecientos noventa y tres. Es decir, el porcentaje de personas que vivían con menos de un dólar por día pasó del treinta al veintinueve por ciento."""
    },
    {
        "source": """- Rentabilidad
INTERÉS DE 2 PESOS SOBRE UNA INVERSIÓN DE 100 PESOS = RENTABILIDAD DE 2 % ((2 ÷ 100) × 100 = 2%) POR CADA 100 PESOS SE GANAN 2 PESOS
 Endeudamiento
 DEUDA DE 30 PESOS POR SOBRE EL CAPITAL DE 20 PESOS = ENDEUDAMIENTO DE 1,5 VECES EL CAPITAL (30 ÷ 20 = 1,5) POR CADA PESO DE CAPITAL EXISTE 1,5 PESOS DE DEUDA
 Liquidez
80 PESOS DE DINERO DISPONIBLE POR SOBRE DEUDAS DE 40 PESOS = LIQUIDEZ DE DOS VECES EL VALOR DE LA DEUDA (80 ÷ 40 = 2) POR CADA PESO DE DEUDA SE DISPONE DE 2 PESOS PARA PAGO""",
        "target": """EL INTERÉS DE DOS PESOS SOBRE UNA INVERSIÓN DE CIEN PESOS DA UNA RENTABILIDAD DE DOS POR CIENTO. ES DECIR, POR CADA CIEN PESOS GANAMOS DOS PESOS. LA DEUDA DE TREINTA PESOS SOBRE EL CAPITAL DE VEINTE PESOS RESULTA EN UNA DEUDA DE UNO PUNTO CINCO VECES EL CAPITAL. ES DECIR, POR CADA PESO DE CAPITAL HAY UNO PUNTO CINCO PESOS DE DEUDA. EN OCHENTA PESOS DE DINERO DISPONIBLE SOBRE DEUDAS DE CUARENTA PESOS DA UNA LIQUIDEZ DEL DOBLE DE LA DEUDA. POR CADA PESO ADEUDADO HAY DOS PESOS PARA PAGAR."""
    },
]

FEW_SHOT_EXAMPLE_IDS = [78, 1805, 3635, 5262]

print("Número de ejemplos few-shot:", len(FEW_SHOT_EXAMPLES))
print("IDs few-shot:", FEW_SHOT_EXAMPLE_IDS)

Número de ejemplos few-shot: 4
IDs few-shot: [78, 1805, 3635, 5262]


In [6]:
PROMPT_TYPE = "few-shot"

FINALIST_CONFIGS = [
    {
        "model_key": "llama3",
        "temperature": 0.2,
        "top_p": 0.85,
        "repetition_penalty": 1.05,
        "max_new_tokens": 256,
        "config_label": "llama3_cfg_1",
    },
    {
        "model_key": "llama3",
        "temperature": 0.2,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
        "config_label": "llama3_cfg_2",
    },
    {
        "model_key": "llama3",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.15,
        "max_new_tokens": 256,
        "config_label": "llama3_cfg_3",
    },
    {
        "model_key": "mistral",
        "temperature": 0.3,
        "top_p": 0.90,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
        "config_label": "mistral_cfg_1",
    },
    {
        "model_key": "mistral",
        "temperature": 0.1,
        "top_p": 0.85,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
        "config_label": "mistral_cfg_2",
    },
    {
        "model_key": "mistral",
        "temperature": 0.2,
        "top_p": 0.85,
        "repetition_penalty": 1.10,
        "max_new_tokens": 256,
        "config_label": "mistral_cfg_3",
    },
]

ACTIVE_RULESETS = ["R0", "R1", "R2", "R3", "R4"]

print("Prompt type fijo:", PROMPT_TYPE)
print("N configuraciones finalistas:", len(FINALIST_CONFIGS))
print("Rulesets:", ACTIVE_RULESETS)
display(pd.DataFrame(FINALIST_CONFIGS))

Prompt type fijo: few-shot
N configuraciones finalistas: 6
Rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']


,model_key,temperature,top_p,repetition_penalty,max_new_tokens,config_label
0,llama3,0.2,0.85,1.05,256,llama3_cfg_1
1,llama3,0.2,0.90,1.10,256,llama3_cfg_2
2,llama3,0.3,0.90,1.15,256,llama3_cfg_3
3,mistral,0.3,0.90,1.10,256,mistral_cfg_1
4,mistral,0.1,0.85,1.10,256,mistral_cfg_2
5,mistral,0.2,0.85,1.10,256,mistral_cfg_3


In [7]:
for cfg in FINALIST_CONFIGS:
    if cfg["model_key"] not in MODELS:
        raise ValueError(f"Modelo no definido en MODELS: {cfg['model_key']}")

for ruleset in ACTIVE_RULESETS:
    if ruleset not in RULESETS:
        raise ValueError(f"Ruleset no definido en RULESETS: {ruleset}")

if PROMPT_TYPE not in ["zero-shot", "few-shot"]:
    raise ValueError(f"Técnica no soportada: {PROMPT_TYPE}")

print("Configuración validada correctamente.")

Configuración validada correctamente.


In [8]:
runner = ExperimentRunner(
    experiment_id=EXPERIMENT_ID,
    log_dir=str(PROJECT_ROOT / "outputs" / "logs")
)

print("Runner inicializado:", runner.experiment_id)

Runner inicializado: exp_ruleset_comparison_20260322_133408


In [9]:
test_row = df_refine.iloc[0]
test_cfg = FINALIST_CONFIGS[0]

test_record = runner.run_one(
    dataset_name="muestra_20_ruleset_comparison",
    model_key=test_cfg["model_key"],
    prompt_type=PROMPT_TYPE,
    ruleset=ACTIVE_RULESETS[0],
    source_text=test_row["source_text"],
    reference_text=test_row["reference_text"],
    sample_id=str(test_row["sample_id"]),
    fold_id=None,
    split_name="ruleset_comparison",
    few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
    few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
    generation_config={
        "temperature": test_cfg["temperature"],
        "top_p": test_cfg["top_p"],
        "repetition_penalty": test_cfg["repetition_penalty"],
        "max_new_tokens": test_cfg["max_new_tokens"],
    },
)

test_record.to_dict()

{'experiment_id': 'exp_ruleset_comparison_20260322_133408',
 'run_id': '8aa9d167-5379-42d8-b9dc-79718454ff07',
 'timestamp': '2026-03-22T13:35:10.464779',
 'dataset_name': 'muestra_20_ruleset_comparison',
 'fold_id': None,
 'split_name': 'ruleset_comparison',
 'model_key': 'llama3',
 'model_id': 'meta-llama/Meta-Llama-3-8B-Instruct',
 'backend': 'ollama',
 'prompt_type': 'few-shot',
 'ruleset': 'R0',
 'few_shot_example_ids': [78, 1805, 3635, 5262],
 'generation_config': {'temperature': 0.2,
  'top_p': 0.85,
  'repetition_penalty': 1.05,
  'max_new_tokens': 256},
 'sample_id': '2088',
 'source_text': 'La comunidad humana más antigua ha sido denominada horda primitiva.',
 'reference_text': 'La comunidad humana más antigua se llamó tribu.',
 'generated_text': 'La comunidad humana más antigua se llama horda primitiva.',
 'prompt_text': 'Reescribe en español cada texto con lenguaje más claro y sencillo.\nConserva el significado original y no inventes información.\n\nDevuelve solo la versión

In [10]:
all_records = []

total_runs = len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)
run_counter = 0

for cfg in FINALIST_CONFIGS:
    for ruleset in ACTIVE_RULESETS:
        for _, row in df_refine.iterrows():
            run_counter += 1

            generation_config = {
                "temperature": cfg["temperature"],
                "top_p": cfg["top_p"],
                "repetition_penalty": cfg["repetition_penalty"],
                "max_new_tokens": cfg["max_new_tokens"],
            }

            print(
                f"[{run_counter}/{total_runs}] "
                f"model={cfg['model_key']} | cfg={cfg['config_label']} | "
                f"ruleset={ruleset} | sample_id={row['sample_id']}"
            )

            record = runner.run_one(
                dataset_name="muestra_20_ruleset_comparison",
                model_key=cfg["model_key"],
                prompt_type=PROMPT_TYPE,
                ruleset=ruleset,
                source_text=row["source_text"],
                reference_text=row["reference_text"],
                sample_id=str(row["sample_id"]),
                fold_id=None,
                split_name="ruleset_comparison",
                few_shot_examples=FEW_SHOT_EXAMPLES if PROMPT_TYPE == "few-shot" else None,
                few_shot_example_ids=FEW_SHOT_EXAMPLE_IDS if PROMPT_TYPE == "few-shot" else None,
                generation_config=generation_config,
            )

            record_dict = record.to_dict()
            record_dict["config_label"] = cfg["config_label"]

            for meta_col in ["idFinal", "grupo", "motivo", "lex"]:
                if meta_col in row.index:
                    record_dict[meta_col] = row[meta_col]

            all_records.append(record_dict)

print(f"Corridas completadas: {len(all_records)}")

[1/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=2088
[2/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=2976
[3/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3430
[4/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3679
[5/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3145
[6/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=507
[7/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=1756
[8/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3093
[9/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3192
[10/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3525
[11/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3627
[12/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3206
[13/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=329
[14/600] model=llama3 | cfg=llama3_cfg_1 | ruleset=R0 | sample_id=3481
[15/600] model=ll

In [11]:
results_df = pd.DataFrame(all_records)

print("Shape results_df:", results_df.shape)
display(results_df.head(3))

if "status" in results_df.columns:
    display(results_df["status"].value_counts(dropna=False))

Shape results_df: (600, 27)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,prompt_text,inference_seconds,status,error_message,metrics,config_label,idFinal,grupo,motivo,lex
0,exp_ruleset_comparison_20260322_133408,6dd5fc71-4b3c-454a-944e-6fd887194638,2026-03-22T13:35:23.019558,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,8.5495,success,None,{},llama3_cfg_1,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_ruleset_comparison_20260322_133408,7f411bb5-f24c-4ff7-8e12-951cd3f517ac,2026-03-22T13:35:31.663336,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,6.2555,success,None,{},llama3_cfg_1,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_ruleset_comparison_20260322_133408,4293f934-b566-4365-9b26-9ca5525aa954,2026-03-22T13:35:38.002786,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,6.3379,success,None,{},llama3_cfg_1,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


status
success    600
Name: count, dtype: int64

In [12]:
raw_results_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_raw_results.csv"
results_df.to_csv(raw_results_path, index=False, encoding="utf-8-sig")

print(raw_results_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_raw_results.csv


In [13]:
successful_df = results_df[results_df["status"] == "success"].copy()

print("Corridas exitosas:", len(successful_df))
print("Corridas con error:", len(results_df) - len(successful_df))
display(successful_df.head(3))

Corridas exitosas: 600
Corridas con error: 0


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,prompt_text,inference_seconds,status,error_message,metrics,config_label,idFinal,grupo,motivo,lex
0,exp_ruleset_comparison_20260322_133408,6dd5fc71-4b3c-454a-944e-6fd887194638,2026-03-22T13:35:23.019558,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,8.5495,success,None,{},llama3_cfg_1,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",1
1,exp_ruleset_comparison_20260322_133408,7f411bb5-f24c-4ff7-8e12-951cd3f517ac,2026-03-22T13:35:31.663336,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,6.2555,success,None,{},llama3_cfg_1,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,1
2,exp_ruleset_comparison_20260322_133408,4293f934-b566-4365-9b26-9ca5525aa954,2026-03-22T13:35:38.002786,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,Reescribe en español cada texto con lenguaje m...,6.3379,success,None,{},llama3_cfg_1,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,1


In [14]:
evaluated_df = evaluate_dataframe(
    successful_df,
    source_col="source_text",
    pred_col="generated_text",
    ref_col="reference_text",
)

print("Shape evaluated_df:", evaluated_df.shape)
display(evaluated_df.head(3))

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

Shape evaluated_df: (600, 52)


,experiment_id,run_id,timestamp,dataset_name,fold_id,split_name,model_key,model_id,backend,prompt_type,...,additions_proportion,deletions_proportion,inflesz_pred,inflesz_src,inflesz_delta,rouge1_f,rouge2_f,rougeL_f,bertscore_f1,sbert_similarity
0,exp_ruleset_comparison_20260322_133408,6dd5fc71-4b3c-454a-944e-6fd887194638,2026-03-22T13:35:23.019558,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.222222,0.300000,52.468333,34.855000,17.613333,0.736842,0.705882,0.736842,0.912748,None
1,exp_ruleset_comparison_20260322_133408,7f411bb5-f24c-4ff7-8e12-951cd3f517ac,2026-03-22T13:35:31.663336,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.416667,0.562500,101.385000,105.172500,-3.787500,0.222222,0.080000,0.222222,0.777562,None
2,exp_ruleset_comparison_20260322_133408,4293f934-b566-4365-9b26-9ca5525aa954,2026-03-22T13:35:38.002786,muestra_20_ruleset_comparison,None,ruleset_comparison,llama3,meta-llama/Meta-Llama-3-8B-Instruct,ollama,few-shot,...,0.352941,0.352941,76.229118,76.229118,0.000000,0.266667,0.071429,0.133333,0.840431,None


In [15]:
param_cols = evaluated_df["generation_config"].apply(pd.Series)
param_cols = param_cols.rename(columns=lambda c: f"gen_{c}")

evaluated_df = pd.concat([evaluated_df, param_cols], axis=1)

display(
    evaluated_df[
        [
            "model_key",
            "config_label",
            "ruleset",
            "gen_temperature",
            "gen_top_p",
            "gen_repetition_penalty",
            "gen_max_new_tokens",
        ]
    ].head(5)
)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens
0,llama3,llama3_cfg_1,R0,0.2,0.85,1.05,256.0
1,llama3,llama3_cfg_1,R0,0.2,0.85,1.05,256.0
2,llama3,llama3_cfg_1,R0,0.2,0.85,1.05,256.0
3,llama3,llama3_cfg_1,R0,0.2,0.85,1.05,256.0
4,llama3,llama3_cfg_1,R0,0.2,0.85,1.05,256.0


In [16]:
evaluated_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_evaluated.csv"
evaluated_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")

print(evaluated_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_evaluated.csv


In [17]:
summary_by_ruleset = summarize_metrics(
    evaluated_df,
    group_cols=[
        "model_key",
        "config_label",
        "ruleset",
        "gen_temperature",
        "gen_top_p",
        "gen_repetition_penalty",
        "gen_max_new_tokens",
    ],
)

display(
    summary_by_ruleset.sort_values(
        by=["model_key", "config_label", "sari", "bertscore_f1"],
        ascending=[True, True, False, False]
    ).head(30)
)

summary_by_ruleset_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_summary_by_ruleset.csv"
summary_by_ruleset.to_csv(summary_by_ruleset_path, index=False, encoding="utf-8-sig")

print(summary_by_ruleset_path)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
2,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,256.0,42.218842,17.534299,86.000712,...,0.0,0.264534,0.443409,0.497984,0.318418,0.443558,71.519639,51.983615,19.536025,0.807977
4,llama3,llama3_cfg_1,R4,0.2,0.85,1.05,256.0,41.435601,15.959837,85.395647,...,0.0,0.290212,0.449717,0.492150,0.290881,0.421004,71.687930,51.983615,19.704316,0.805806
3,llama3,llama3_cfg_1,R3,0.2,0.85,1.05,256.0,40.885400,15.258565,86.790510,...,0.0,0.289702,0.476705,0.472744,0.282990,0.403528,73.182701,51.983615,21.199087,0.800278
1,llama3,llama3_cfg_1,R1,0.2,0.85,1.05,256.0,40.858116,15.578270,83.063042,...,0.0,0.300491,0.475684,0.469121,0.290455,0.405819,68.215495,51.983615,16.231880,0.802526
0,llama3,llama3_cfg_1,R0,0.2,0.85,1.05,256.0,40.141862,15.430296,87.049805,...,0.0,0.247064,0.428208,0.496590,0.296089,0.422793,72.072198,51.983615,20.088583,0.808021
5,llama3,llama3_cfg_2,R0,0.2,0.90,1.10,256.0,43.080279,15.378169,81.424226,...,0.0,0.325193,0.469978,0.489988,0.299905,0.411665,66.517505,51.983615,14.533891,0.803035
6,llama3,llama3_cfg_2,R1,0.2,0.90,1.10,256.0,41.371788,13.733112,83.202532,...,0.0,0.348406,0.502785,0.464178,0.273831,0.391600,68.276101,51.983615,16.292486,0.798941
7,llama3,llama3_cfg_2,R2,0.2,0.90,1.10,256.0,41.219090,13.408150,83.175336,...,0.0,0.387777,0.526703,0.459837,0.257257,0.382569,68.496923,51.983615,16.513308,0.792370
8,llama3,llama3_cfg_2,R3,0.2,0.90,1.10,256.0,41.114241,13.818544,81.776082,...,0.0,0.369931,0.530286,0.451392,0.279060,0.380980,69.200101,51.983615,17.216487,0.796844
9,llama3,llama3_cfg_2,R4,0.2,0.90,1.10,256.0,39.419634,13.293801,83.710383,...,0.0,0.385819,0.520273,0.423883,0.248194,0.369649,69.724713,51.983615,17.741098,0.790714


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_summary_by_ruleset.csv


In [18]:
best_ruleset_per_config = (
    summary_by_ruleset
    .sort_values(
        by=["model_key", "config_label", "sari", "bertscore_f1"],
        ascending=[True, True, False, False]
    )
    .groupby(["model_key", "config_label"], as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(best_ruleset_per_config)

best_ruleset_per_config_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_best_ruleset_per_config.csv"
best_ruleset_per_config.to_csv(best_ruleset_per_config_path, index=False, encoding="utf-8-sig")

print(best_ruleset_per_config_path)

,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,gen_max_new_tokens,sari,bleu,fernandez_huerta_pred,...,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
0,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,256.0,42.218842,17.534299,86.000712,...,0.0,0.264534,0.443409,0.497984,0.318418,0.443558,71.519639,51.983615,19.536025,0.807977
1,llama3,llama3_cfg_2,R0,0.2,0.90,1.10,256.0,43.080279,15.378169,81.424226,...,0.0,0.325193,0.469978,0.489988,0.299905,0.411665,66.517505,51.983615,14.533891,0.803035
2,llama3,llama3_cfg_3,R1,0.3,0.90,1.15,256.0,42.880052,14.297969,82.165161,...,0.0,0.391925,0.481467,0.477212,0.284198,0.417246,64.355483,51.983615,12.371868,0.807136
3,mistral,mistral_cfg_1,R4,0.3,0.90,1.10,256.0,40.401948,13.345582,82.633741,...,0.0,0.477569,0.331331,0.456189,0.262416,0.405492,49.876043,51.983615,-2.107572,0.794802
4,mistral,mistral_cfg_2,R3,0.1,0.85,1.10,256.0,40.434464,12.624830,82.895772,...,0.0,0.494952,0.354683,0.434542,0.240833,0.375357,55.999944,51.983615,4.016329,0.791387
5,mistral,mistral_cfg_3,R1,0.2,0.85,1.10,256.0,40.270143,12.054415,82.196194,...,0.0,0.483938,0.344280,0.459635,0.273581,0.402421,56.125854,51.983615,4.142240,0.792454


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_best_ruleset_per_config.csv


In [19]:
best_ruleset_per_model = summarize_metrics(
    evaluated_df,
    group_cols=["model_key", "ruleset"]
)

display(
    best_ruleset_per_model.sort_values(
        by=["model_key", "sari", "bertscore_f1"],
        ascending=[True, False, False]
    )
)

best_ruleset_per_model_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_best_ruleset_per_model.csv"
best_ruleset_per_model.to_csv(best_ruleset_per_model_path, index=False, encoding="utf-8-sig")

print(best_ruleset_per_model_path)

,model_key,ruleset,sari,bleu,fernandez_huerta_pred,fernandez_huerta_src,fernandez_huerta_delta,compression_ratio_eval,sentence_splits,levenshtein_similarity,exact_copy,additions_proportion,deletions_proportion,rouge1_f,rouge2_f,rougeL_f,inflesz_pred,inflesz_src,inflesz_delta,bertscore_f1
1,llama3,R1,41.703319,14.536451,82.810245,88.314342,-5.504097,0.799636,0.633333,0.507955,0.0,0.346941,0.486645,0.470170,0.282828,0.404888,66.949026,51.983615,14.965412,0.802867
0,llama3,R0,41.626767,14.701496,84.181949,88.314342,-4.132393,0.791796,0.633333,0.531421,0.0,0.311662,0.464031,0.484691,0.287697,0.404479,69.139928,51.983615,17.156313,0.802757
2,llama3,R2,41.608226,14.533838,83.597401,88.314342,-4.716941,0.785014,0.633333,0.495928,0.0,0.350931,0.499221,0.465772,0.274919,0.398111,69.328446,51.983615,17.344831,0.798241
3,llama3,R3,40.873464,13.548733,84.391800,88.314342,-3.922542,0.761928,0.683333,0.499727,0.0,0.363476,0.520509,0.446496,0.265952,0.375481,71.484233,51.983615,19.500618,0.794576
4,llama3,R4,40.178607,13.535948,84.171244,88.314342,-4.143098,0.784052,0.633333,0.498136,0.0,0.376268,0.514206,0.440426,0.254669,0.375508,69.931854,51.983615,17.948239,0.793289
8,mistral,R3,40.230126,12.143225,83.466995,88.314342,-4.847347,1.546852,0.750000,0.398159,0.0,0.497135,0.343421,0.437703,0.248891,0.383648,54.531481,51.983615,2.547867,0.787769
6,mistral,R1,40.027850,11.790851,83.408301,88.314342,-4.906041,1.474459,0.933333,0.443986,0.0,0.483521,0.346800,0.448513,0.261325,0.399799,57.779057,51.983615,5.795442,0.791853
9,mistral,R4,39.757337,11.799485,85.482036,88.314342,-2.832306,1.517613,0.950000,0.393332,0.0,0.502646,0.348552,0.430888,0.239237,0.377814,56.690355,51.983615,4.706740,0.785692
7,mistral,R2,39.547305,11.882960,82.313738,88.314342,-6.000604,1.350670,0.750000,0.443981,0.0,0.469570,0.359741,0.456846,0.261156,0.401277,56.552035,51.983615,4.568420,0.794422
5,mistral,R0,39.507746,13.331363,83.445128,88.314342,-4.869214,1.270979,0.700000,0.471520,0.0,0.433382,0.350316,0.462134,0.273595,0.408877,57.342996,51.983615,5.359381,0.795401


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_best_ruleset_per_model.csv


In [20]:
selected_configs = best_ruleset_per_config[
    ["model_key", "config_label", "ruleset"]
].drop_duplicates()

finalist_cases = evaluated_df.merge(
    selected_configs,
    on=["model_key", "config_label", "ruleset"],
    how="inner"
)

qual_cols = [
    "sample_id",
    "idFinal",
    "grupo",
    "motivo",
    "model_key",
    "config_label",
    "ruleset",
    "gen_temperature",
    "gen_top_p",
    "gen_repetition_penalty",
    "source_text",
    "reference_text",
    "generated_text",
    "sari",
    "bertscore_f1",
    "rougeL_f",
    "compression_ratio_eval",
    "exact_copy",
]

qual_cols = [c for c in qual_cols if c in finalist_cases.columns]

qual_review_df = finalist_cases[qual_cols].copy()

display(qual_review_df.head(20))

qual_review_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_qualitative_review.csv"
qual_review_df.to_csv(qual_review_path, index=False, encoding="utf-8-sig")

print(qual_review_path)

,sample_id,idFinal,grupo,motivo,model_key,config_label,ruleset,gen_temperature,gen_top_p,gen_repetition_penalty,source_text,reference_text,generated_text,sari,bertscore_f1,rougeL_f,compression_ratio_eval,exact_copy
0,2088,1872_LibroBAC.pdf,A_cortos,"Muy corto, sustitución léxica clara.",llama3,llama3_cfg_1,R2,0.2,0.85,1.05,La comunidad humana más antigua ha sido denomi...,La comunidad humana más antigua se llamó tribu.,La comunidad humana más antigua se llamó horda...,77.410414,0.924922,0.842105,0.900000,0
1,2976,881_LibroNEFE_Sincopyright.pdf,A_cortos,Corto con dinero/cantidad.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,Prueba a poner en un sobre o frasco $1 por día...,Pon en un sobre un dólar diario más las moneda...,Prueba a guardar cada día una moneda de un dól...,35.914269,0.801748,0.303030,1.000000,0
2,3430,2692_LibroUide_Sincopyright.pdf,A_cortos,Frase conceptual simple de negocios.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,Uno de los problemas más importantes en los ne...,Asignar precios a los productos es un problema...,Uno de los problemas más grandes en los negoci...,25.832848,0.839404,0.133333,1.000000,0
3,3679,829_LibroUAC_Sincopyright.pdf,A_cortos,Corto con número e IVA.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,"El resultado será 190, o sea, el IVA del kilo ...","El resultado será ciento noventa. Es decir, el...",El IVA del kilo de pan es de $190.,32.491125,0.751794,0.344828,0.600000,0
4,3145,1045_LibroUide_Sincopyright.pdf,A_cortos,Corto con proporción financiera.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,"Por cada $100 facturados, la compañía gasta $2...",La empresa gasta dos dólares con veinte centav...,"Por cada $100 que la empresa factura, gasta $2...",27.679739,0.740161,0.222222,1.090909,0
5,507,562_LibroUAC_Sincopyright.pdf,B_medianos,Porcentajes y rentabilidad.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,"En general, se dice que la rentabilidad normal...","Generalmente, la rentabilidad normal de una in...",La rentabilidad normal de una inversión en paí...,60.983625,0.921257,0.800000,0.708333,0
6,1756,5100_LibroBAC.pdf,B_medianos,Redacción abstracta/institucional.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,En virtud de estas y otras reflexiones fue que...,Debido a estas y otras reflexiones se diseñó e...,Este capítulo fue diseñado para ser parte del ...,35.343017,0.750624,0.350000,0.600000,0
7,3093,875_LibroUide_Sincopyright.pdf,B_medianos,Definición contable.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,Las cuentas de activo reflejan aquello que pos...,Las cuentas de activo evidencian las pertenenc...,Las cuentas de activo muestran lo que una orga...,62.022704,0.855775,0.424242,0.937500,0
8,3192,1202_LibroUide_Sincopyright.pdf,B_medianos,"Relación de pasivo/activo, útil para precisión.",llama3,llama3_cfg_1,R2,0.2,0.85,1.05,"Por cada dólar de pasivo corriente, la compañí...",La empresa tiene noventa y tres centavos de dó...,La empresa tiene 93 centavos de dólar en activ...,58.174377,0.922097,0.809524,0.842105,0
9,3525,230_LibroUAC_Sincopyright.pdf,B_medianos,Dato económico con cifra.,llama3,llama3_cfg_1,R2,0.2,0.85,1.05,Posee la renta per cápita más alta de Latinoam...,Tiene la renta per cápita más alta de Latinoam...,La renta per cápita en Latinoamérica es la más...,37.970401,0.788622,0.465116,0.904762,0


/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_qualitative_review.csv


In [21]:
bitacora = {
    "experiment_id": EXPERIMENT_ID,
    "dataset_name": "muestra_20_ruleset_comparison",
    "n_samples": int(len(df_refine)),
    "prompt_type_fixed": PROMPT_TYPE,
    "finalist_configs": FINALIST_CONFIGS,
    "rulesets_tested": ACTIVE_RULESETS,
    "expected_total_runs": int(len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)),
    "successful_runs": int(len(successful_df)),
    "leader_metrics": ["sari", "bertscore_f1"],
    "support_metrics": ["rougeL_f", "compression_ratio_eval", "exact_copy"],
    "best_ruleset_per_config": best_ruleset_per_config.to_dict(orient="records"),
}

bitacora_path = OUTPUT_DIR / f"{EXPERIMENT_ID}_bitacora_experimento.json"

with open(bitacora_path, "w", encoding="utf-8") as f:
    json.dump(bitacora, f, ensure_ascii=False, indent=2)

print(bitacora_path)

/home/harielpadillasanchez/Documentos/TT/TT2/outputs/ruleset_comparison/exp_ruleset_comparison_20260322_133408_bitacora_experimento.json


In [22]:
print(f"""
Resumen rapido de esta fase:

- Se probaron {len(FINALIST_CONFIGS)} configuraciones finalistas.
- Se fijo la tecnica en: {PROMPT_TYPE}
- Se compararon {len(ACTIVE_RULESETS)} rulesets: {ACTIVE_RULESETS}
- En total se esperaban {len(FINALIST_CONFIGS) * len(ACTIVE_RULESETS) * len(df_refine)} corridas.

El objetivo de esta fase fue ver como cambia el desempeño cuando se modifica el conjunto de reglas,
manteniendo fijas las configuraciones que ya habian sido seleccionadas en la etapa de hiperparametros.
La comparacion se hizo principalmente con SARI y BERTScore F1, apoyandose tambien en otras metricas
para revisar el balance entre simplificacion, preservacion del significado y nivel de compresion.
""")


Resumen rapido de esta fase:

- Se probaron 6 configuraciones finalistas.
- Se fijo la tecnica en: few-shot
- Se compararon 5 rulesets: ['R0', 'R1', 'R2', 'R3', 'R4']
- En total se esperaban 600 corridas.

El objetivo de esta fase fue ver como cambia el desempeño cuando se modifica el conjunto de reglas,
manteniendo fijas las configuraciones que ya habian sido seleccionadas en la etapa de hiperparametros.
La comparacion se hizo principalmente con SARI y BERTScore F1, apoyandose tambien en otras metricas
para revisar el balance entre simplificacion, preservacion del significado y nivel de compresion.

